# Clase 7 — Tu chatbot como servicio web con Flask

## Objetivos de aprendizaje
- Entender la arquitectura cliente-servidor de una aplicación de IA generativa.
- Explicar para qué sirve CORS y por qué hace falta en un chatbot web.
- Reconocer las rutas `GET /` y `POST /chatbot` de un servidor Flask real.
- Levantar un servidor Flask dentro del propio notebook y probarlo con peticiones HTTP reales.
- Identificar la limitación del historial de conversación global compartido.

## Qué vamos a construir
Vamos a retomar el chatbot Blenderbot de la clase 6 y exponerlo como un servicio Flask real
—las mismas rutas `GET /` y `POST /chatbot` del proyecto guiado de IBM
(`LLM_application_chatbot-main/app.py`)— corriendo el servidor en un hilo dentro de esta
misma notebook, para poder probarlo con peticiones HTTP sin necesidad de exponer un puerto
a internet.

## La arquitectura cliente-servidor, en corto

- **Backend (Flask)** = la cocina: tiene el modelo cargado y sabe generar respuestas.
- **Frontend (HTML/JS)** = el camarero: lleva el mensaje del usuario y trae la respuesta.
- **CORS** = el permiso explícito para aceptar pedidos de un origen distinto al del servidor.

Repasa la analogía completa y la limitación del historial global compartido en el
[Capítulo 11 del manual](../manual/11_chatbot_web_flask.md).

## El `app.py` real del proyecto guiado (referencia)

Así es el backend Flask completo del proyecto `LLM_application_chatbot-main` de IBM (código
real). No lo ejecutamos tal cual en Colab porque depende de `templates/index.html`, pero es
exactamente lo que vamos a reproducir en las celdas de código siguientes:

```python
from flask import Flask, request, render_template
from flask_cors import CORS
import json
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

app = Flask(__name__)
CORS(app)

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
conversation_history = []

@app.route('/', methods=['GET'])
def home():
    return render_template('index.html')

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_data(as_text=True)
    data = json.loads(data)
    input_text = data['prompt']
    history = "\n".join(conversation_history)
    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")
    outputs = model.generate(**inputs)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    conversation_history.append(input_text)
    conversation_history.append(response)
    return response

if __name__ == '__main__':
    app.run()
```

Y el frontend (`static/script.js`) deja el endpoint del chatbot sin resolver, a propósito:

```javascript
const url = 'www.example.com';  // Make a POST request to this url
```

Vas a resolver ese mismo placeholder en la práctica de esta clase.

## Cómo lo probamos dentro de Colab

Google Colab no te permite levantar un servidor Flask visible al público con un clic, pero sí
puedes correr Flask **dentro del propio proceso de Python**, escuchando en `127.0.0.1`
(localhost), usando un hilo (`threading.Thread`) para que el servidor corra en paralelo sin
bloquear la celda. Luego le hacemos peticiones HTTP reales con la librería `requests`, desde
la misma notebook. Es exactamente el mismo servidor Flask del proyecto real, solo que no lo
exponemos a internet.

In [ ]:
# Instalamos Flask y flask-cors (requests ya viene instalado en Colab)
!pip install -q flask flask-cors requests

In [ ]:
# Reutilizamos el mismo chatbot Blenderbot de la clase 6
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Modelo Blenderbot cargado.")

In [ ]:
import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # permite peticiones desde cualquier origen (ver capítulo 11 del manual)

# Historial de conversación: una única lista global (ver la limitación
# de diseño que explicamos en el capítulo 11 del manual)
conversation_history = []

@app.route('/', methods=['GET'])
def home():
    # En el proyecto real esto serviría templates/index.html;
    # aquí devolvemos un texto simple porque no tenemos frontend en Colab
    return "Servidor Flask del chatbot activo. Usa POST /chatbot con {'prompt': '...'}"

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    input_text = data['prompt']

    # Igual que en app.py: unimos el historial en un string
    history = "\n".join(conversation_history)

    # Tokenizamos historial + mensaje nuevo y generamos la respuesta
    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=60)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Guardamos el intercambio en el historial global
    conversation_history.append(input_text)
    conversation_history.append(response)

    return jsonify({"response": response})

def run_flask_app():
    # use_reloader=False es importante: el recargador automático de Flask
    # no funciona bien dentro de un hilo de Colab
    app.run(port=5000, use_reloader=False)

# Lanzamos el servidor Flask en un hilo en segundo plano, sin bloquear la notebook
server_thread = threading.Thread(target=run_flask_app, daemon=True)
server_thread.start()

time.sleep(3)  # damos tiempo a que el servidor termine de arrancar
print("Servidor Flask corriendo en http://127.0.0.1:5000")

In [ ]:
import requests

resp = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "Hola, ¿cómo estás?"}
)
print("Status:", resp.status_code)
print("Respuesta:", resp.json())

In [ ]:
# Enviamos un segundo mensaje para ver cómo el historial global se va acumulando
resp2 = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "¿Cuál es tu color favorito?"}
)
print("Respuesta:", resp2.json())

# El historial global ahora contiene los dos intercambios anteriores.
# Si "otro usuario" llamara a este mismo servidor ahora mismo, su mensaje
# se mezclaría en el mismo historial: esta es la limitación de diseño real
# que explicamos en el capítulo 11 del manual.
print("\nHistorial acumulado (compartido por todas las peticiones):")
print(conversation_history)

## Resumen de la clase
- Un chatbot web separa el backend (Flask + modelo) del frontend (HTML/JS), comunicados por HTTP.
- `flask_cors` habilita peticiones desde otros orígenes.
- La ruta `POST /chatbot` recibe el prompt, genera la respuesta y actualiza el historial.
- En Colab, probamos el servidor real corriéndolo en un hilo sobre `127.0.0.1` y llamándolo con `requests`, sin necesidad de exponerlo a internet.
- El historial de conversación es una lista global compartida por todas las peticiones: una limitación real que hay que tener en cuenta.

Sigue con la [práctica de esta clase](../notebooks_practica/clase07_practica_chatbot_web_flask.ipynb)
y repasa el [Capítulo 11 del manual](../manual/11_chatbot_web_flask.md) si quieres profundizar.